# Memory: GRU over frame sequences (subject-independent protocol, all DMD participants)

**Question:** does memory (the last 2 seconds) add anything over the per-frame hybrid model? (Section 6.2, Table 6 of the paper.)

The GRU receives a sequence of `SEQ` frames (10 frames = 2 s at 5 fps) and outputs the state in the last frame. The model sees only the past, as in a car.

**Variants (per-frame inputs):**
- **G**: EAR, MAR, head angles (as in model A, but without hand-made windows: the GRU learns the dynamics);
- **GB**: G + 52 MediaPipe blendshapes (blinks, mouth, gaze direction);
- **GBE**: GB + MobileNetV2 embeddings (PCA 32), a temporal version of the hybrid;
- **P**: normalised coordinates of 90 eye, iris and lip points.

Every variant has a "face found" flag, so the model handles frames without a face itself (no fixed rule) and they count in the metrics.

**Protocol:** calibration on the first 10 s of each recording without labels; scaling and PCA are fitted on the training people only; the number of epochs was fixed in advance (no early stopping on the test set). Seeds 0, 1, 2; results are averaged.

**Inputs on Drive** (`MyDrive/driver_project/`): `dmd_for_colab.zip` (only labels.csv is used), `landmarks.zip` (folder from 02b), and for GBE `results/emb_all_v2.npy` (created by m0b_honest_colab.ipynb).

**Run:** T4 GPU, set `TASK`, Run all.

In [ ]:
import os, glob, subprocess, time
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEV)

## Settings

In [ ]:
TASK = "attention"          # "drowsy" or "attention"
CONFIGS = ["G", "GB", "GBE", "P"]

SEQ = 10                 # frames per sequence (2 s at 5 fps)
CALIB_SEC, FPS = 10, 5
HIDDEN, EPOCHS, BATCH, LR = 64, 15, 512, 1e-3
SEEDS = [0, 1, 2]

ROOT = "/content/drive/MyDrive/driver_project"
DATA_ZIP, LM_ZIP = f"{ROOT}/dmd_for_colab.zip", f"{ROOT}/landmarks.zip"
EMB_CACHE, OUT_DIR = f"{ROOT}/results/emb_all_v2.npy", f"{ROOT}/results"
DATA_DIR = "/content/data"

## Data

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.exists(f"{DATA_DIR}/labels.csv"):
    subprocess.run(["unzip", "-q", "-o", DATA_ZIP, "labels.csv", "-d", DATA_DIR], check=True)
if not os.path.exists(f"{DATA_DIR}/landmarks"):
    subprocess.run(["unzip", "-q", "-o", LM_ZIP, "-d", DATA_DIR], check=True)
print("npz files:", len(glob.glob(f"{DATA_DIR}/landmarks/*.npz")))

## Per-frame features
All frames of each recording in order (frames without the task label serve as context).

In [ ]:
BASE = ["ear", "mar", "yaw", "pitch", "roll"]
TASKS = {"drowsy": ("drowsy_label", ["alert", "eyes_closed", "yawning"]),
         "attention": ("attention_label", ["looking_road", "not_looking_road"])}
label_col, classes = TASKS[TASK]

def load_tables(data_dir, emb_cache):
    lab = pd.read_csv(f"{data_dir}/labels.csv", keep_default_na=False, na_values=[""], low_memory=False)
    for c in BASE:
        lab[c] = pd.to_numeric(lab[c], errors="coerce")
    lab["face_found"] = pd.to_numeric(lab["face_found"], errors="coerce").fillna(0).astype(int)
    lab = lab.sort_values(["video_id", "frame_idx"]).reset_index(drop=True)

    # embeddings: same order as in m0b_honest (frames with a face, sorted by video_id, frame_idx)
    E = None
    fi = np.flatnonzero(lab["face_found"].values == 1)
    if os.path.exists(emb_cache):
        E_all = np.load(emb_cache)
        if len(E_all) == len(fi):
            E = np.zeros((len(lab), E_all.shape[1]), np.float32); E[fi] = E_all
        else:
            print(f"Embedding cache length does not match ({len(E_all)} != {len(fi)}): GBE skipped")
    else:
        print("No embedding cache: GBE skipped (run m0b_honest_colab.ipynb first)")

    # landmarks and blendshapes from 02b
    B = np.full((len(lab), 52), np.nan, np.float32)
    P = np.full((len(lab), 180), np.nan, np.float32)
    lm_face = np.zeros(len(lab), np.int8)
    pos = {k: i for i, k in enumerate(zip(lab["video_id"], lab["frame_idx"]))}
    missing = 0
    for vid in lab["video_id"].unique():
        fn = f"{data_dir}/landmarks/{vid}.npz"
        if not os.path.exists(fn):
            missing += 1; continue
        d = np.load(fn)
        rows = np.array([pos.get((vid, int(f)), -1) for f in d["frame_idx"]])
        ok = rows >= 0
        rows = rows[ok]
        pts = d["pts"][ok]
        ids = list(d["point_ids"])
        nose, le, re = ids.index(1), ids.index(33), ids.index(263)
        xy = pts[:, :, :2] - pts[:, nose:nose + 1, :2]                          # centre: nose tip
        scale = np.linalg.norm(pts[:, le, :2] - pts[:, re, :2], axis=1)[:, None, None]
        P[rows] = (xy / (scale + 1e-6)).reshape(len(rows), -1)                 # scale: inter-ocular distance
        B[rows] = d["blend"][ok][:, :52]
        lm_face[rows] = d["face"][ok]
    print("Recordings without a landmark file:", missing)
    return lab, E, B, P, lm_face

lab, E, B, P, lm_face = load_tables(DATA_DIR, EMB_CACHE)
print("frames:", len(lab), "| people:", lab["person"].nunique())

In [ ]:
def calibrate(X, vids, valid, n_cal):
    """Subtract from each recording the median over its first n_cal valid frames (no labels)."""
    X = X.copy()
    for v in np.unique(vids):
        m = np.flatnonzero((vids == v) & valid)
        if len(m):
            X[vids == v] -= np.nanmedian(X[m[:n_cal]], axis=0)
    return X

n_cal = CALIB_SEC * FPS
vids = lab["video_id"].values
face = lab["face_found"].values == 1
G = lab[BASE].values.astype(np.float32)
G = pd.DataFrame(G).groupby(vids).ffill(limit=3).values.astype(np.float32)
G_ok = face & ~np.isnan(G).any(1)
Gc = calibrate(G, vids, G_ok, n_cal)
B_ok = lm_face == 1
Bc = calibrate(B, vids, B_ok, n_cal)
Pc = calibrate(P, vids, B_ok, n_cal)
Ec = calibrate(E, vids, face, n_cal) if E is not None else None

def blocks(cfg):
    """Feature blocks: (matrix, validity mask). Embeddings are separate: they need PCA in every fold."""
    out = []
    if "G" in cfg: out.append((Gc, G_ok))
    if "B" in cfg: out.append((Bc, B_ok))
    if cfg == "P": out.append((Pc, B_ok))
    return out

## Sequences and model

In [ ]:
# positions of target frames (labelled with the task classes)
y_name = lab[label_col].values
target = np.isin(y_name, classes)
y_all = np.array([classes.index(c) if c in classes else -1 for c in y_name])
person = lab["person"].values
# start of each recording, so a window never crosses into another video
start = pd.Series(np.arange(len(lab))).groupby(vids).transform("min").values

def windows(idx):
    """Frame indices of the window [t-SEQ+1 .. t] for each t; -1 where the window starts before the recording."""
    w = idx[:, None] - np.arange(SEQ - 1, -1, -1)[None, :]
    return np.where(w >= start[idx][:, None], w, -1)

class GRUNet(nn.Module):
    def __init__(self, d, n_out):
        super().__init__()
        self.gru = nn.GRU(d, HIDDEN, batch_first=True)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(HIDDEN, n_out))
    def forward(self, x):
        h, _ = self.gru(x)
        return self.head(h[:, -1])

def make_X(F, idx):
    """(len(idx), SEQ, d): window features, unavailable frames are zeros (validity flags are inside F)."""
    w = windows(idx)
    X = F[np.clip(w, 0, None)]
    X[w < 0] = 0
    return X

def frame_matrix(cfg, tr_mask):
    """Features of all frames for one fold: standardisation (and embedding PCA) on training people only."""
    parts = []
    for X, ok in blocks(cfg):
        mu = np.nanmean(X[tr_mask & ok], 0); sd = np.nanstd(X[tr_mask & ok], 0) + 1e-6
        Z = np.where(ok[:, None], (X - mu) / sd, 0).astype(np.float32)
        parts += [np.nan_to_num(Z), ok[:, None].astype(np.float32)]
    if "E" in cfg:
        fit = tr_mask & face
        sub = np.flatnonzero(fit)
        sub = sub[np.random.RandomState(0).permutation(len(sub))[:20000]]
        mu = Ec[sub].mean(0); sd = Ec[sub].std(0) + 1e-6
        pca = PCA(32, random_state=0).fit((Ec[sub] - mu) / sd)
        Z = pca.transform((Ec - mu) / sd).astype(np.float32)
        Z = Z / (Z[fit].std(0) + 1e-6)
        parts += [np.where(face[:, None], Z, 0).astype(np.float32), face[:, None].astype(np.float32)]
    return np.concatenate(parts, 1)

def train_predict(F, tr_idx, te_idx, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    Xtr = torch.tensor(make_X(F, tr_idx)); ytr = torch.tensor(y_all[tr_idx])
    Xte = torch.tensor(make_X(F, te_idx))
    cnt = np.bincount(y_all[tr_idx], minlength=len(classes))
    w = torch.tensor(len(tr_idx) / (len(classes) * np.maximum(cnt, 1)), dtype=torch.float32, device=DEV)
    net = GRUNet(F.shape[1], len(classes)).to(DEV)
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=1e-4)
    lossf = nn.CrossEntropyLoss(weight=w)
    for ep in range(EPOCHS):
        net.train()
        perm = torch.randperm(len(Xtr))
        for b in range(0, len(perm), BATCH):
            j = perm[b:b + BATCH]
            opt.zero_grad()
            loss = lossf(net(Xtr[j].to(DEV)), ytr[j].to(DEV))
            loss.backward(); opt.step()
    net.eval(); out = []
    with torch.no_grad():
        for b in range(0, len(Xte), 4096):
            out.append(net(Xte[b:b + 4096].to(DEV)).argmax(1).cpu().numpy())
    return np.concatenate(out)

## Training: one person in test, the rest in training

In [ ]:
persons = sorted(set(person[target]))
cfgs = [c for c in CONFIGS if not ("E" in c and Ec is None)]
preds = {c: {s: np.full(len(lab), -1) for s in SEEDS} for c in cfgs}
t0 = time.time()
for cfg in cfgs:
    for p in persons:
        tr_mask = person != p
        F = frame_matrix(cfg, tr_mask)
        tr_idx = np.flatnonzero(target & tr_mask)
        te_idx = np.flatnonzero(target & ~tr_mask)
        for s in SEEDS:
            preds[cfg][s][te_idx] = train_predict(F, tr_idx, te_idx, s)
    print(f"{cfg}: done ({(time.time()-t0)/60:.1f} min)")

## Results
Macro-F1 per person (all labelled frames, including frames without a face), averaged over seeds; mean and SD over people.

In [ ]:
REF_A = {"drowsy": 0.874, "attention": 0.816}[TASK]   # geometry baseline A (04b) on the same people
yawn = lab["yawning"].astype(str).values if "yawning" in lab.columns else None
rows = []
for cfg in cfgs:
    row, per = {"config": cfg}, []
    for p in persons:
        m = target & (person == p)
        f = [f1_score(y_all[m], preds[cfg][s][m], labels=sorted(set(y_all[m])), average="macro", zero_division=0)
             for s in SEEDS]
        row[p] = round(float(np.mean(f)), 3); per.append(np.mean(f))
    row["mean"] = round(float(np.mean(per)), 3)
    row["std_people"] = round(float(np.std(per, ddof=1)), 3)
    row["std_seeds"] = round(float(np.std([np.mean([f1_score(y_all[target & (person == p)],
                                 preds[cfg][s][target & (person == p)],
                                 labels=sorted(set(y_all[target & (person == p)])), average="macro",
                                 zero_division=0) for p in persons]) for s in SEEDS])), 3)
    if TASK == "drowsy" and yawn is not None:
        pr = preds[cfg][SEEDS[0]]
        iy = target & (y_all == classes.index("yawning"))
        for t, short in [("Yawning with hand", "yawn_with_hand_%"), ("Yawning without hand", "yawn_no_hand_%")]:
            mt = iy & (yawn == t)
            row[short] = round(100 * float((pr[mt] == classes.index("yawning")).mean()), 1) if mt.any() else np.nan
        row["false_yawn_frames"] = int(((pr == classes.index("yawning")) & target & ~iy).sum())
    rows.append(row)
table = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print(table.to_string(index=False))
print(f"\nFor comparison: A (geometry + 1 s window, random forest) = {REF_A}")

In [ ]:
table.to_csv(f"{OUT_DIR}/summary_gru_{TASK}.csv", index=False)
out = lab.loc[target, ["video_id", "frame_idx", "person", label_col]].copy()
for cfg in cfgs:
    out[f"pred_{cfg}"] = [classes[i] for i in preds[cfg][SEEDS[0]][target]]
out.to_csv(f"{OUT_DIR}/preds_gru_{TASK}.csv", index=False)
print("Saved to", OUT_DIR)

In [ ]:
from scipy.stats import wilcoxon
g = pd.read_csv(f"{OUT_DIR}/summary_gru_attention.csv").set_index("config")
m = pd.read_csv(f"{OUT_DIR}/summary_m0b_honest_attention.csv").set_index("config")
pp = [c for c in g.columns if c.startswith("g")]
e = m.loc["E_hybrid (D + A)", pp].astype(float)
for cfg in ["GBE", "P"]:
    x = g.loc[cfg, pp].astype(float)
    print(cfg, "- E:", round((x - e).mean(), 4), "| better for", (x > e).sum(), "of", len(pp), "|", wilcoxon(x, e))